# 📘 Notebook 6: Encapsulation — Getters, Setters & Properties

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Understand **encapsulation** and why it matters
2. Use **naming conventions** for access control (`_protected`, `__private`)
3. Create **properties** with the `@property` decorator
4. Implement **getters**, **setters**, and **deleters**
5. Understand **name mangling** for double-underscore attributes
6. Apply validation logic in setters

---

## 1. What is Encapsulation?

**Encapsulation** is the bundling of data (attributes) and the methods that operate on that data into a single unit (class), while **restricting direct access** to some of the object's components.

### Real-World Analogy 🏦

Think of a **bank account**:
- You can't directly modify the bank's database to change your balance
- You must use controlled interfaces: `deposit()`, `withdraw()`, `check_balance()`
- These methods enforce rules (can't withdraw more than balance, can't deposit negative amounts)

### Why Encapsulate?

| Benefit | Description |
|---------|-------------|
| **Data Protection** | Prevent invalid modifications to internal state |
| **Controlled Access** | Add validation, logging, or transformation on get/set |
| **Implementation Freedom** | Change internal structure without breaking external code |
| **Reduced Complexity** | Hide implementation details, expose simple interface |

---

## 2. Python's Access Control Conventions

Unlike languages like Java or C++ that have `private` and `protected` keywords, Python uses **naming conventions**:

| Convention | Syntax | Meaning | Enforcement |
|-----------|--------|---------|-------------|
| **Public** | `self.name` | Accessible everywhere | None — fully open |
| **Protected** | `self._name` | "Please don't touch" (convention only) | ⚠️ Honor system |
| **Private** | `self.__name` | Name-mangled to prevent accidental access | 🔒 Name mangling |

> 💡 Python follows the philosophy: **"We are all consenting adults."** There's no true `private` — it's about communicating intent.

In [ ]:
class Employee:
    def __init__(self, name: str, salary: float, ssn: str):
        self.name = name          # Public — anyone can access
        self._salary = salary     # Protected — "internal use, don't touch"
        self.__ssn = ssn          # Private — name-mangled

emp = Employee("Alice", 85000, "123-45-6789")

# Public — works fine
print(f"Name: {emp.name}")

# Protected — works, but signals "don't access directly"
print(f"Salary: ${emp._salary}")

# Private — AttributeError!
try:
    print(emp.__ssn)
except AttributeError as e:
    print(f"❌ Can't access __ssn directly: {e}")

# But it's not truly private — Python name-mangles it
print(f"SSN via mangled name: {emp._Employee__ssn}")
print("↑ Never do this in real code!")

### Name Mangling Explained

When you write `self.__name`, Python internally renames it to `self._ClassName__name`:

```python
self.__ssn        →  self._Employee__ssn
self.__price      →  self._Item__price
self.__balance    →  self._BankAccount__balance
```

This prevents **accidental** access, not **intentional** access. It's mainly to avoid naming conflicts in inheritance.

In [ ]:
class Employee:
    def __init__(self, name, salary, ssn):
        self.name = name
        self._salary = salary
        self.__ssn = ssn

emp = Employee("Alice", 85000, "123-45-6789")

# See what's actually stored
print("Object's __dict__:")
for key, value in emp.__dict__.items():
    print(f"  {key}: {value}")

# Notice: __ssn became _Employee__ssn

---

## 3. The Problem: Direct Attribute Access

Without encapsulation, anyone can set attributes to invalid values:

In [ ]:
# WITHOUT encapsulation — no protection!
class ItemBad:
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

item = ItemBad("Phone", 100)

# Anyone can set invalid values!
item.price = -999     # Negative price?!
item.name = ""        # Empty name?!

print(f"Price: ${item.price}")  # -999 — that's wrong!
print(f"Name: '{item.name}'")

---

## 4. The `@property` Decorator

The `@property` decorator turns a method into a **"getter"** — it looks like an attribute but runs a method behind the scenes.

```python
class Item:
    @property
    def name(self):       # Getter: item.name (no parentheses!)
        return self.__name
    
    @name.setter
    def name(self, value): # Setter: item.name = "Phone"
        self.__name = value
```

The beauty: **external code uses `item.name` as if it's a simple attribute**, but internally it's calling methods with validation.

In [ ]:
class Item:
    def __init__(self, name: str, price: float):
        self.__name = name      # Private storage
        self.__price = price    # Private storage
    
    # === GETTER for name ===
    @property
    def name(self):
        """Read-only access to the item name."""
        return self.__name
    
    # === SETTER for name (with validation) ===
    @name.setter
    def name(self, value: str):
        if not isinstance(value, str):
            raise TypeError("Name must be a string!")
        if len(value) > 50:
            raise ValueError("Name is too long (max 50 characters)!")
        if len(value) == 0:
            raise ValueError("Name cannot be empty!")
        self.__name = value
    
    # === GETTER for price ===
    @property
    def price(self):
        """Read-only access to the item price."""
        return self.__price
    
    # === SETTER for price (with validation) ===
    @price.setter
    def price(self, value: float):
        if value < 0:
            raise ValueError(f"Price cannot be negative! Got {value}")
        self.__price = value
    
    def __repr__(self):
        return f"Item('{self.__name}', ${self.__price})"


# Usage looks EXACTLY like regular attributes!
item = Item("Phone", 100)
print(f"Name: {item.name}")     # Calls the getter
print(f"Price: ${item.price}")  # Calls the getter

# Setting values — calls the setter with validation
item.name = "Smartphone"        # Valid ✅
item.price = 150                 # Valid ✅
print(f"\nUpdated: {item}")

# Invalid values — setter rejects them!
print("\nTrying invalid values:")
try:
    item.price = -50
except ValueError as e:
    print(f"  ❌ {e}")

try:
    item.name = ""
except ValueError as e:
    print(f"  ❌ {e}")

try:
    item.name = 12345
except TypeError as e:
    print(f"  ❌ {e}")

---

## 5. Read-Only Properties

If you define a **getter but no setter**, the property becomes **read-only**:

In [ ]:
class Circle:
    def __init__(self, radius: float):
        self.__radius = radius
    
    @property
    def radius(self):
        return self.__radius
    
    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("Radius must be positive!")
        self.__radius = value
    
    @property
    def area(self):
        """Computed property — read-only, no setter defined."""
        import math
        return math.pi * self.__radius ** 2
    
    @property
    def circumference(self):
        """Computed property — read-only."""
        import math
        return 2 * math.pi * self.__radius


c = Circle(5)
print(f"Radius: {c.radius}")
print(f"Area: {c.area:.2f}")               # Computed on access — always up to date!
print(f"Circumference: {c.circumference:.2f}")

# Change radius — computed properties automatically update
c.radius = 10
print(f"\nNew radius: {c.radius}")
print(f"New area: {c.area:.2f}")            # Automatically recalculated!

# Can't set computed properties
try:
    c.area = 100  # No setter defined!
except AttributeError as e:
    print(f"\n❌ Can't set area: {e}")

---

## 6. The `@property.deleter` Decorator

You can also control what happens when `del` is used on a property:

In [ ]:
class User:
    def __init__(self, username: str, email: str):
        self.__username = username
        self.__email = email
    
    @property
    def email(self):
        return self.__email
    
    @email.setter
    def email(self, value):
        if '@' not in value:
            raise ValueError("Invalid email address!")
        self.__email = value
    
    @email.deleter
    def email(self):
        print(f"⚠️ Deleting email for {self.__username}")
        self.__email = None
    
    def __repr__(self):
        return f"User('{self.__username}', email='{self.__email}')"


user = User("alice", "alice@example.com")
print(f"Email: {user.email}")

user.email = "alice.new@example.com"  # Setter with validation
print(f"Updated: {user.email}")

del user.email  # Calls the deleter
print(f"After delete: {user}")

---

## 7. Full Encapsulation Example: Bank Account

In [ ]:
class BankAccount:
    """A fully encapsulated bank account."""
    
    _interest_rate = 0.02  # Protected class attribute
    
    def __init__(self, owner: str, initial_balance: float = 0):
        self.__owner = owner              # Private — can't change owner
        self.__balance = initial_balance   # Private — only modify via methods
        self.__transaction_history = []    # Private — internal tracking
    
    # === Read-only property: owner ===
    @property
    def owner(self):
        """Account owner (read-only after creation)."""
        return self.__owner
    
    # === Read-only property: balance ===
    @property
    def balance(self):
        """Current balance (read-only — use deposit/withdraw)."""
        return self.__balance
    
    # === Read-only property: transaction count ===
    @property
    def transaction_count(self):
        return len(self.__transaction_history)
    
    # === Controlled methods for modifying balance ===
    def deposit(self, amount: float):
        """Deposit money — enforces positive amounts."""
        if amount <= 0:
            raise ValueError(f"Deposit amount must be positive! Got ${amount}")
        self.__balance += amount
        self.__transaction_history.append(f"Deposit: +${amount}")
        print(f"✅ Deposited ${amount}. Balance: ${self.__balance}")
    
    def withdraw(self, amount: float):
        """Withdraw money — enforces sufficient funds."""
        if amount <= 0:
            raise ValueError(f"Withdrawal amount must be positive! Got ${amount}")
        if amount > self.__balance:
            raise ValueError(f"Insufficient funds! Balance: ${self.__balance}, Requested: ${amount}")
        self.__balance -= amount
        self.__transaction_history.append(f"Withdrawal: -${amount}")
        print(f"✅ Withdrew ${amount}. Balance: ${self.__balance}")
    
    def apply_interest(self):
        """Apply interest to the balance."""
        interest = self.__balance * self._interest_rate
        self.__balance += interest
        self.__transaction_history.append(f"Interest: +${interest:.2f}")
        print(f"✅ Interest applied: +${interest:.2f}. Balance: ${self.__balance:.2f}")
    
    def get_statement(self):
        """Print a summary of all transactions."""
        print(f"\n{'='*40}")
        print(f"Account Statement for {self.__owner}")
        print(f"{'='*40}")
        for i, transaction in enumerate(self.__transaction_history, 1):
            print(f"  {i}. {transaction}")
        print(f"{'='*40}")
        print(f"  Current Balance: ${self.__balance:.2f}")
        print(f"{'='*40}")
    
    def __repr__(self):
        return f"BankAccount(owner='{self.__owner}', balance=${self.__balance:.2f})"


# Use the encapsulated account
account = BankAccount("Alice", 1000)
account.deposit(500)
account.withdraw(200)
account.apply_interest()

# Can't directly modify balance!
try:
    account.balance = 1000000  # ← No setter — read-only!
except AttributeError as e:
    print(f"\n❌ Can't set balance directly: {e}")

# Can't directly modify owner!
try:
    account.owner = "Evil Hacker"
except AttributeError as e:
    print(f"❌ Can't change owner: {e}")

# View the statement
account.get_statement()

---

## 8. Properties with Inheritance

Properties work correctly with inheritance. Child classes can override properties:

In [ ]:
class Item:
    def __init__(self, name: str, price: float):
        self.__name = name
        self.__price = price
    
    @property
    def price(self):
        return self.__price
    
    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("Price must be non-negative!")
        self.__price = value
    
    @property
    def name(self):
        return self.__name
    
    def apply_discount(self):
        self.price = self.price * self.pay_rate


class PremiumItem(Item):
    """Premium items have a luxury tax."""
    pay_rate = 0.9  # Only 10% discount
    luxury_tax = 0.15
    
    @property
    def price_with_tax(self):
        """Price including luxury tax."""
        return self.price * (1 + self.luxury_tax)


regular = Item("Widget", 10)
premium = PremiumItem("Gold Watch", 5000)

print(f"Regular: ${regular.price}")
print(f"Premium: ${premium.price} (before tax)")
print(f"Premium: ${premium.price_with_tax} (after {premium.luxury_tax*100}% luxury tax)")

---

## 🏋️ Practice Exercises

### Exercise 1: `Password` class
Create a `Password` class with:
- Private `__value` attribute
- `@property` getter that returns masked password (e.g., `****`)
- `@property.setter` that validates: min 8 chars, at least one digit, at least one uppercase
- A `verify(attempt)` method that checks if the attempt matches

In [ ]:
# Exercise 1: Your code here



### Exercise 2: `Temperature` with Properties
Create a `Temperature` class with:
- Private `__celsius` attribute
- `celsius` property with getter and setter (validates >= -273.15)
- `fahrenheit` property with getter AND setter (conversion both ways)
- `kelvin` read-only property

In [ ]:
# Exercise 2: Your code here



---

### ✅ Solutions

In [ ]:
# Solution 1: Password class
class Password:
    def __init__(self, value: str):
        self.value = value  # Uses the setter for validation
    
    @property
    def value(self):
        """Returns masked password."""
        return '*' * len(self.__value)
    
    @value.setter
    def value(self, new_value: str):
        if len(new_value) < 8:
            raise ValueError("Password must be at least 8 characters!")
        if not any(c.isdigit() for c in new_value):
            raise ValueError("Password must contain at least one digit!")
        if not any(c.isupper() for c in new_value):
            raise ValueError("Password must contain at least one uppercase letter!")
        self.__value = new_value
    
    def verify(self, attempt: str) -> bool:
        return self.__value == attempt

# Test
pwd = Password("MyPass123")
print(f"Masked: {pwd.value}")
print(f"Verify correct: {pwd.verify('MyPass123')}")
print(f"Verify wrong: {pwd.verify('wrong')}")

try:
    pwd.value = "short"
except ValueError as e:
    print(f"❌ {e}")

In [ ]:
# Solution 2: Temperature with dual-access properties
class Temperature:
    def __init__(self, celsius: float):
        self.celsius = celsius  # Uses setter
    
    @property
    def celsius(self):
        return self.__celsius
    
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"{value}°C is below absolute zero!")
        self.__celsius = value
    
    @property
    def fahrenheit(self):
        return (self.__celsius * 9/5) + 32
    
    @fahrenheit.setter
    def fahrenheit(self, value):
        """Set temperature via Fahrenheit — converts to Celsius internally."""
        self.celsius = (value - 32) * 5/9  # Uses celsius setter for validation
    
    @property
    def kelvin(self):
        return self.__celsius + 273.15
    
    def __repr__(self):
        return f"Temperature({self.__celsius:.1f}°C / {self.fahrenheit:.1f}°F / {self.kelvin:.1f}K)"

# Test
t = Temperature(100)
print(f"Set to 100°C:  {t}")

t.fahrenheit = 32  # Set via Fahrenheit!
print(f"Set to 32°F:   {t}")

t.celsius = -40  # -40 is the same in both scales
print(f"Set to -40°C:  {t}")

---

## 📌 Key Takeaways

1. **Encapsulation** = bundling data + controlling access to it
2. **`_single_underscore`** = "protected" (convention: don't access from outside)
3. **`__double_underscore`** = "private" (name-mangled: `_ClassName__attr`)
4. **`@property`** creates getters that look like attributes: `obj.name`
5. **`@name.setter`** adds validation when setting: `obj.name = "value"`
6. **Read-only properties** = getter without setter (computed/protected values)
7. Properties let you **change implementation** without breaking external code

---

**⏭️ Next: [Notebook 07 — The Four Pillars of OOP](./07_Four_Pillars_of_OOP.ipynb)** — Master Encapsulation, Abstraction, Inheritance, and Polymorphism!